# 04 - Model Evaluation
RMSE, MAE, F1, SSI, Wasserstein Distance

In [1]:
!pip install -q tensorflow scikit-learn scipy

In [2]:
from google.colab import drive
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, mean_absolute_error, f1_score
from scipy.stats import wasserstein_distance

drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/fishing_project/"

Mounted at /content/drive


In [3]:
# Load model and test data
model = tf.keras.models.load_model(DATA_DIR + 'convlstm_model.h5')

# Load test data (saved from 03_model_training.ipynb)
X_test = np.load(DATA_DIR + 'X_test.npy')
y_test = np.load(DATA_DIR + 'y_test.npy')

print(f"Test data: {X_test.shape} → {y_test.shape}")
# Expected: (20, 3, 41, 25, 6) → (20, 41, 25, 1)

Test data: (11, 3, 41, 25, 7) → (11, 41, 25, 1)


In [4]:
# Generate predictions
# Fill NaN inputs (land/coastal cells) with 0 before predicting
X_test_clean = np.nan_to_num(X_test, nan=0.0)
preds = model.predict(X_test_clean)
print(f"Predictions shape: {preds.shape}")

# Save predictions for 05_visualization.ipynb
np.save(DATA_DIR + 'predictions.npy', preds)

# Flatten and force-clean any remaining NaN → 0
y_flat = np.nan_to_num(y_test.flatten(), nan=0.0)
p_flat = np.nan_to_num(preds.flatten(), nan=0.0)

print(f"NaN in y_flat: {np.isnan(y_flat).sum()}")
print(f"NaN in p_flat: {np.isnan(p_flat).sum()}")



1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Predictions shape: (11, 41, 25, 1)
NaN in y_flat: 0
NaN in p_flat: 0


In [5]:
# 1. RMSE & MAE
rmse = np.sqrt(mean_squared_error(y_flat, p_flat))
mae  = mean_absolute_error(y_flat, p_flat)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")


RMSE: 0.207062
MAE:  0.157864


In [6]:
# 2. F1 Score (binarized at threshold 0.5)
y_binary    = (y_flat > 0.5).astype(int)
pred_binary = (p_flat > 0.5).astype(int)

f1 = f1_score(y_binary, pred_binary, zero_division=0)
print(f"F1 Score: {f1:.4f}")


F1 Score: 0.0000


In [7]:
# 3. SSI (Structural Similarity Index)
def calculate_ssi(obs, pred):
    """Compute SSI per sample"""
    C1, C2 = 0.01**2, 0.03**2
    mu_obs, mu_pred     = obs.mean(), pred.mean()
    sigma_obs, sigma_pred = obs.std(), pred.std()
    sigma_cross = np.mean((obs - mu_obs) * (pred - mu_pred))

    luminance = (2 * mu_obs * mu_pred + C1) / (mu_obs**2 + mu_pred**2 + C1)
    contrast  = (2 * sigma_obs * sigma_pred + C2) / (sigma_obs**2 + sigma_pred**2 + C2)
    structure = (sigma_cross + C2/2) / (sigma_obs * sigma_pred + C2/2)

    return luminance * contrast * structure

ssi_scores = []
for i in range(len(y_test)):
    yi = y_test[i].flatten()
    pi = preds[i].flatten()
    m  = ~(np.isnan(yi) | np.isnan(pi))
    ssi_scores.append(calculate_ssi(yi[m], pi[m]))

ssi_mean = np.mean(ssi_scores)
ssi_std  = np.std(ssi_scores)
print(f"SSI: {ssi_mean:.4f} ± {ssi_std:.4f}")


SSI: 0.1150 ± 0.0164


In [8]:
# 4. Wasserstein Distance
wd_scores = []
for i in range(len(y_test)):
    obs_flat  = y_test[i].flatten()
    pred_flat = preds[i].flatten()
    m = ~(np.isnan(obs_flat) | np.isnan(pred_flat))
    obs_flat, pred_flat = obs_flat[m], pred_flat[m]

    # Avoid division by zero if all zeros
    obs_sum  = obs_flat.sum()
    pred_sum = pred_flat.sum()
    if obs_sum == 0 or pred_sum == 0:
        wd_scores.append(np.nan)
        continue

    obs_dist  = obs_flat / obs_sum
    pred_dist = pred_flat / pred_sum

    wd = wasserstein_distance(
        np.arange(len(obs_dist)),
        np.arange(len(pred_dist)),
        obs_dist,
        pred_dist
    )
    wd_scores.append(wd)

wd_scores = [s for s in wd_scores if not np.isnan(s)]
wd_mean = np.mean(wd_scores)
wd_std  = np.std(wd_scores)
print(f"Wasserstein Distance: {wd_mean:.4f} ± {wd_std:.4f}")


Wasserstein Distance: 122.4656 ± 40.1295


In [9]:
# Create results table
results = pd.DataFrame({
    'Metric': ['RMSE', 'MAE', 'F1', 'SSI', 'Wasserstein'],
    'Value': [
        f"{rmse:.3e}",
        f"{mae:.3e}",
        f"{f1:.4f}",
        f"{ssi_mean:.4f}",
        f"{wd_mean:.4f}"
    ],
    'Std Dev': [
        'N/A', 'N/A', 'N/A',
        f"{ssi_std:.4f}",
        f"{wd_std:.4f}"
    ]
})

print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
print(results.to_string(index=False))

results.to_csv(DATA_DIR + 'evaluation_results.csv', index=False)
print("\n✔ Evaluation complete")



EVALUATION RESULTS
     Metric     Value Std Dev
       RMSE 2.071e-01     N/A
        MAE 1.579e-01     N/A
         F1    0.0000     N/A
        SSI    0.1150  0.0164
Wasserstein  122.4656 40.1295

✅ Evaluation complete
